# Pandas 高级功能

### 1. 数据合并与连接

1. `pd.merge(left_df, right_df, how, on='col', left_on, right_on, suffixes)`

`left_df` : 左侧DataFrame

`right_df` : 右侧DataFrame

`how` : 合并方式，`inner`, `outer`, `left`, `right`

`on` : 连接的列名 (如果两侧列明不同，可以用 `left_on` 和 `right_on`)

`left_on` : 左侧DataFrame的连接列

`right_on` : 右侧DataFrame的连接列

`suffixes` : 添加后缀，以区分重复的列名

In [3]:
import pandas as pd

left = pd.DataFrame({'ID': [1, 2, 3], 'Name': ['Alice', 'Bob', 'Charlie']})
right = pd.DataFrame({'ID': [1, 2, 4], 'Age': [24, 27, 22]})

# 使用merge进行内连接
result1 = pd.merge(left, right, on='ID', how='inner')
display(result1)
# 外连接
result2 = pd.merge(left, right, on='ID', how='outer')
display(result2)
# 左连接
result3 = pd.merge(left, right, on='ID', how='left')
display(result3)
# 右连接
result4 = pd.merge(left, right, on='ID', how='right')
display(result4)

,ID,Name,Age
0,1,Alice,24
1,2,Bob,27


,ID,Name,Age
0,1,Alice,24.0
1,2,Bob,27.0
2,3,Charlie,NaN
3,4,NaN,22.0


,ID,Name,Age
0,1,Alice,24.0
1,2,Bob,27.0
2,3,Charlie,NaN


,ID,Name,Age
0,1,Alice,24
1,2,Bob,27
2,4,NaN,22


### 2. `concat()` -- 沿轴连接

`concat()` 用于对各DataFrame沿指定轴(行或列)进行连接，常用于行合并(垂直连接)或列合并(水平连接)

`pd.concat(objs[obj1, obj2], axis=0/1, ignore_index=False/True, keys)`

`objs` : 需要合并的 DataFrame 列表

`axis` : 合并的轴，0表示按行合并，1表示按列合并

`ignore_index` : 是否忽略索引，重新生成索引(默认 `False`)

`keys` : 为合并的对象提供层次化索引

In [8]:
df1 = pd.DataFrame({'A': [1, 2, 3]})
df2 = pd.DataFrame({'A': [4, 5, 6]})

# 沿横向轴合并
df_concat1 = pd.concat([df1, df2], axis=0, ignore_index=False)
display(df_concat1)
# 沿纵向轴合并
df_concat2 = pd.concat([df1, df2], axis=1, ignore_index=True)
display(df_concat2)

,A
0,1
1,2
2,3
0,4
1,5
2,6


,0,1
0,1,4
1,2,5
2,3,6


### 3. `join()` -- 基于索引连接

简化连接操作, 通常用于基于索引将多个DataFrame连接

`df1.join(other=df2, how='inner/outer/left/right, on='col')`

In [9]:
left = pd.DataFrame({'A': [1, 2, 3]}, index=[1, 2, 3])
right = pd.DataFrame({'B': [4, 5, 6]}, index=[1, 2, 4])

df_joined = left.join(right, how='inner')
display(df_joined)

,A,B
1,1,4
2,2,5


# 二、透视表和交叉表

透视表 : `pd.pivot_table()`，适用于多个维度

交叉表 : `pd.crosstab()`，适用于两个维度

透视表和交叉表用于数据的汇总和重新排列

1. `pd.pivot_table(data, values='col1', index='col2', columns='col3', aggfunc=mean(default), fill_value=)`

`data` : 要输入的数据 DataFrame

`values` : 汇总的列

`index` : 用作行索引的列

`column` : 用作列索引的列

`aggfunc` : 聚合函数，默认为 `mean` , 也可以是 `sum`, `count`等

`fill_value` : 填充缺失值

2. `pd.crosstab(index, columns, values, aggfunc='count', margins=True/False)`

`margins` : 每行每列自动求和

In [15]:
data = {'Date': ['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04'],
        'Category': ['A', 'B', 'A', 'B'],
        'Sales': [100, 150, 200, 250]}
df = pd.DataFrame(data)

# 透视表
pivot_table = pd.pivot_table(df, values='Sales', index=['Date'], columns=['Category'], aggfunc='sum', fill_value=0)
print(pivot_table)

data = {'Category': ['A', 'B', 'A', 'B', 'A', 'B'],
        'Region': ['North', 'South', 'North', 'South', 'West', 'East']}
df = pd.DataFrame(data)

# 交叉表
cross_table = pd.crosstab(index=df['Category'], columns=df['Region'], margins=True)
print(cross_table)

Category      A    B
Date                
2024-01-01  100    0
2024-01-02    0  150
2024-01-03  200    0
2024-01-04    0  250
Region    East  North  South  West  All
Category                               
A            0      2      0     1    3
B            1      0      2     0    3
All          1      2      2     1    6


# 自定义函数

1. `.apply()` --应用函数到 DataFrame 或 Series 上

在 DataFrame 或 Series 上应用自定义函数，支持对行或列进行操作

df['col'].apply(function, axis, raw, result_type)

| 参数           | 说明                                  |
|--------------|-------------------------------------|
| func         | 	需要应用的函数                            |
| axis         | 	默认为 0，表示按列应用；1 表示按行应用              |
| raw          | 	是否传递原始数据（默认为 False）                |
| result_type	 | 定义输出的类型，如 `expand`, `reduce`, `broadcast` |

2. `.applymap()`  -- 在整个DataFrame上应用函数,性能差，已弃用，不推荐，推荐 `df.transform()`

只能应用于 DataFrame，作用于 DataFrame 中的每个元素

`df.applymap(function)`

3. `.map()` -- 在Series应用函数

可以对 Series 中的每个元素应用一个函数或一个映射关系

`df['col'].map(arg)`

`arg` : 应用的函数，字典或Series，与每个元素对应



#### 方案：用向量化操作（最优选择）
如果操作可向量化（如算术运算、字符串方法），直接用向量化语法效率最高，无需循环或函数：  
```python
df **2  # 直接对所有元素平方，比 transform() 更快
df['a'].str.upper()  # 字符串列的向量化操作
```


### 3. 总结：三者的正确使用场景
| 数据结构 | 需求                  | 推荐函数          | 替代关系               |
|----------|-----------------------|-------------------|------------------------|
| Series   | 元素级操作            | `map()`           | -                      |
| DataFrame| 元素级操作            | `transform()`     | 替代 `applymap()`       |
| DataFrame| 行/列级操作（按轴）   | `apply()`         | -                      |
| 任何结构 | 可向量化的简单操作    | 直接用向量化语法  | 效率远高于上述所有方法  |

记住：能用向量化操作就优先用，其次用 `transform()`（DataFrame）或 `map()`（Series），尽量避免 `apply()` 做元素级操作（性能差）。

In [18]:
df = pd.DataFrame({'A': [1, 2, 3, 4], 'B': [10, 20, 30, 40]})

def custom_func(x):
        return x * 2

display(df)
# apply
df['A'] = df['A'].apply(custom_func)
display(df)
# applymap(已经弃用)
df = df.applymap(lambda x: x ** 2)
display(df)
# map
df = pd.DataFrame({'A': ['cat', 'dog', 'rabbit']})
display(df)
df['A'] = df['A'].map({'cat': 'kitten', 'dog': 'puppy'})
display(df)

,A,B
0,1,10
1,2,20
2,3,30
3,4,40


,A,B
0,2,10
1,4,20
2,6,30
3,8,40


C:\Users\yuanx\AppData\Local\Temp\ipykernel_3644\4008419912.py:11: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x ** 2)


,A,B
0,4,100
1,16,400
2,36,900
3,64,1600


,A
0,cat
1,dog
2,rabbit


,A
0,kitten
1,puppy
2,NaN


# 四、分组与聚合

Pandas 中的 `groupby()` 方法非常强大，可以用于分组聚合、转换数据和过滤数据。通过 `groupby()`，可以将数据根据某些条件分组，进行聚合运算，如求和、均值、计数等。

1. 分组 : `df.groupby(by='col', axis, level)`

`level` : 按照索引的级别进行分组（适用于 MultiIndex）

2. 聚合 : `.agg(func)`

In [26]:
df = pd.DataFrame({
    'Category': ['A', 'B', 'A', 'B', 'A', 'B'],
    'Value': [10, 20, 30, 40, 50, 60]
})
display(df)
# 按照 Category 列进行分组并计算每组的总和
grouped1 = df.groupby(by=['Category'])['Value'].sum()
display(grouped1)
grouped2 = df.groupby(by=['Category']).sum()
display(grouped2)
# 聚合
grouped3 = df.groupby(by=['Category'])['Value'].agg(['sum', 'min', 'max', 'mean'])
display(grouped3)

,Category,Value
0,A,10
1,B,20
2,A,30
3,B,40
4,A,50
5,B,60


Category
A     90
B    120
Name: Value, dtype: int64

,Value
Category,
A,90
B,120


,sum,min,max,mean
Category,,,,
A,90,10,50,30.0
B,120,20,60,40.0


# 五、时间序列处理

日期解析、频率转换、日期范围生成、时间窗口操作

### 1. date_range() -- 生成时间序列

`pd.date_range(start, end, periods=number, freq='d/h')`

start	起始日期  
end	结束日期  
periods	生成的时间点数,即生成多少个时间点  
freq	频率（如 d 表示天，h 表示小时等）

### 2. 日期和时间的偏移

`pd.Timedelta()`

### 3. 时间窗口操作(Rolling, Expanding)

`rolling()` 和 `expanding()` 方法进行滚动和扩展窗口操作，常用于时间序列中的移动平均等计算。

`df.rolling(window=number)` : 计算滚动窗口操作，常用于移动平均等

`df.expanding()` : 计算扩展窗口操作，累计值

In [46]:
date_range = pd.date_range(start='2025-8-31', periods=5, freq='d')
display(date_range)
print(date_range[0:3])

DatetimeIndex(['2025-08-31', '2025-09-01', '2025-09-02', '2025-09-03',
               '2025-09-04'],
              dtype='datetime64[ns]', freq='D')

DatetimeIndex(['2025-08-31', '2025-09-01', '2025-09-02'], dtype='datetime64[ns]', freq='D')


# 六、缺失值处理

isna() : 检查缺失值，返回布尔值

fillna() : 填充缺失值

dropna() : 删除包含缺失值的行或列

# 七、多重索引(MultiIndex)

Pandas 提供了多重索引（MultiIndex）的功能，使得可以在 DataFrame 或 Series 中处理复杂的数据结构，尤其适用于 **层次化** 的数据。通过多重索引，我们能够对数据进行分组、选择、切片以及聚合等操作。

### 1. 创建多重索引

1. `pd.MultiIndex.from_tuples(tuples, names)`: 使用元组来创建多重索引，每个元组对应一个索引层级

tuples	每个元组对应一个索引值  
names	每个索引级别的名称（可选）

2. `pd.MultiIndex.from_product(iterables, name)`: 使用多个列表的笛卡尔积来创建多重索引，适合用于数据维度较多的情况。

iterables	多个列表或数组  
names	每个索引级别的名称（可选）

3. `set_index(['col1', 'col2'], inplace=True)`: 将 DataFrame 的列转换为多重索引，适用于从已有的数据创建多重索引

In [50]:
# 创建元组
index_tuple = [('A', 1), ('A', 2), ('B', 1), ('B', 2)]

# 创建多重索引
multi_index = pd.MultiIndex.from_tuples(index_tuple, names=['Letter', 'Number'])

df = pd.DataFrame({'Value': [10, 20, 30, 40]}, index=multi_index)
print(df)

               Value
Letter Number       
A      1          10
       2          20
B      1          30
       2          40


In [49]:
# 创建多个列表
index_values = [['A', 'B'], [1, 2]]

# 创建多重索引
multi_index = pd.MultiIndex.from_product(index_values, names=['Letter', 'Number'])

df = pd.DataFrame({'Value': [10, 20, 30, 40]}, index=multi_index)
print(df)

               Value
Letter Number       
A      1          10
       2          20
B      1          30
       2          40


In [55]:
data = {
    'Letter': ['A', 'A', 'B', 'B'],
    'Number': [1, 2, 1, 2],
    'Value': [10, 20, 30, 40]
}

df = pd.DataFrame(data)
print(df)
df.set_index(['Letter', 'Number'], inplace=True)
print(df)

  Letter  Number  Value
0      A       1     10
1      A       2     20
2      B       1     30
3      B       2     40
               Value
Letter Number       
A      1          10
       2          20
B      1          30
       2          40


### 2. 多重索引的操作

1. 访问多重索引数据

可以通过层级索引来访问数据。通过 `loc[]` 或 `xs()`（cross-section）可以方便地进行数据选择

2. 多重索引的切片

Pandas 支持对多重索引进行切片操作，允许通过 `loc['']` 索引级别选择不同的子集。

3. 排序多重索引

Pandas 的 `sort_index()` 方法支持对多重索引进行排序。

In [63]:
data = {
    'Letter': ['A', 'A', 'B', 'B'],
    'Number': [1, 2, 1, 2],
    'Value': [10, 20, 30, 40]
}

df = pd.DataFrame(data)

# 设置多重索引
df.set_index(['Letter', 'Number'], inplace=True)

print(df)

# 选择'A'类别，所有'Number'为 1 的数据
display(df.loc['A', 1])
"""
使用 df.xs()获取交叉数据 : 可以在多重索引中选择指定级别的切片
"""
# 使用xs获取'Number' 为 1 的所有数据
print(df.xs(1, level='Number'))
# 选择 Letter 为'A'的所有数据
print(df.loc['A'])
# 按照多重索引排序
df_sorted = df.sort_index(level=['Letter', 'Number'], ascending=[True, False])
print(df_sorted)
# 对多重索引进行聚合
df_grouped = df.groupby(['Letter', 'Number']).sum()
print(df_grouped)
# 将多重索引重置为普通的列
df_reset = df.reset_index()
print(df_reset)
# 多重索引的缺失值
data = {
    'Letter': ['A', 'A', 'B', 'B'],
    'Number': [1, 2, 1, 2],
    'Value': [10, None, 30, 40]
}

df = pd.DataFrame(data)
df.set_index(['Letter', 'Number'], inplace=True)

# 填充缺失值
df_filled = df.fillna(0)
print(df_filled)

               Value
Letter Number       
A      1          10
       2          20
B      1          30
       2          40


Value    10
Name: (A, 1), dtype: int64

        Value
Letter       
A          10
B          30
        Value
Number       
1          10
2          20
               Value
Letter Number       
A      2          20
       1          10
B      2          40
       1          30
               Value
Letter Number       
A      1          10
       2          20
B      1          30
       2          40
  Letter  Number  Value
0      A       1     10
1      A       2     20
2      B       1     30
3      B       2     40
               Value
Letter Number       
A      1        10.0
       2         0.0
B      1        30.0
       2        40.0
